# US3.8 — Robustness / Overfitting Gate

**Frozen validation only.** No result from this notebook may promote or replace the US3.5 six candidates or three controls. Primary model is REAL 1% participation; US3.7 pending-order handling and US3.5 corporate-event corrections are inherited. The 55/45, beta exit, and session-frequency grids below are diagnostic surfaces only.

This notebook is a separate work product. It will not modify US3.5/3.6/3.7 result files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, pandas as pd, os, runpy
ROOT = Path('/content/drive/MyDrive/미국주식데이터')
OUT = ROOT / 'US38_robustness_gate_20260928'
OUT.mkdir(exist_ok=True)
required = ['descriptions.csv','actions.csv','metrics.csv','stocks.csv','tickers.csv','sp500.csv','funds.csv']
missing = [x for x in required if not (ROOT/x).exists()]
assert not missing, f'Missing required universe files: {missing}'
print('Drive mounted; all 7 universe CSVs present')
print({x:(ROOT/x).stat().st_size for x in required})
assert (ROOT/'US35_followup_20260928'/'simulation_complete.json').exists()
assert (ROOT/'US35_followup_20260928'/'summary.csv').exists()
assert (ROOT/'US33_validation_20260927'/'canonical_feature_inputs.parquet').exists()
manifest = json.loads((ROOT/'manifest.json').read_text()) if (ROOT/'manifest.json').exists() else None
print('Source gates: US33 canonical inputs + US35 results present; manifest:', bool(manifest))

In [ ]:
# Reuse the frozen US3.5 simulator source, with only these fixed US3.8 diagnostic surfaces added.
# The copied runner writes only to US38_robustness_gate_20260928.
runner = OUT.parent/'run_us38_surfaces.py'
assert runner.exists(), f'Upload run_us38_surfaces.py into {OUT}'
%pip -q install duckdb==1.3.2 pandas==2.2.3 pyarrow==23.0.1
runpy.run_path(str(runner), run_name='__main__')

In [ ]:
# Candidate/control parity against the immutable US3.5 daily output.
baseline_map = {
  'A0_quarter':'A0_quarter', 'A0_bimonth':'A0_bimonth',
  'A2_quarter':'A2_quarter', 'A2_week':'A2_week',
  'B3_w55':'B3_w55', 'B3_beta60d3':'B3_beta60d3',
  'A0_event':'A0_event', 'A2_event':'A2_event', 'B3_w50':'B3_w50'
}
checks=[]
for new, old in baseline_map.items():
    a=pd.read_csv(OUT/f'daily_v3_{new}.csv')
    b=pd.read_csv(ROOT/'US35_followup_20260928'/f'daily_v3_{old}.csv')
    same_dates=a.date.astype(str).equals(b.date.astype(str))
    err=float((a['return']-b['return']).abs().max()) if same_dates else float('inf')
    checks.append({'config':new,'same_dates':same_dates,'max_daily_return_abs_error':err,'pass_1e-10':same_dates and err<1e-10})
parity=pd.DataFrame(checks)
parity.to_csv(OUT/'us38_baseline_parity_us35.csv',index=False)
display(parity)
assert parity['pass_1e-10'].all(), 'US3.5 parity failure; stop robustness interpretation.'
print('US3.5 candidate/control parity PASS')

In [ ]:
# Write diagnostic-only parameter surfaces. No ranking or candidate promotion.
summary=pd.read_csv(OUT/'summary.csv')
summary.to_csv(OUT/'us38_surface_results.csv',index=False)
core=summary[summary.config.isin(['B3_w45','B3_w50','B3_w55','B3_w60'])].copy()
beta=summary[summary.config.str.match(r'B3_beta(55|60|65)d(2|3|5)$')].copy()
rebalance=summary[summary.config.str.match(r'(A0|A2)_d(05|10|20|40|60)$')].copy()
core.to_csv(OUT/'core_weight_surface.csv',index=False)
beta.to_csv(OUT/'beta_exit_surface.csv',index=False)
rebalance.to_csv(OUT/'trading_day_rebalance_surface.csv',index=False)
display(core[['config','period','CAGR','MDD','Sharpe','annualTurnover']])
display(beta[['config','period','CAGR','MDD','Sharpe','annualTurnover']])
display(rebalance[['config','period','CAGR','MDD','Sharpe','annualTurnover']])
print('US38 surface diagnostics complete; all values remain non-promotable.')

## Reproducibility notes

The runner uses `b4d47858bccd5b732b88af9b14368ac9581e770a`, US3.5 correction revision 3, mapping v2, end date 2026-09-23, one-way cost 25 bp, ADV floor $500k, 20 active days, integer shares, minimum order $10, settlement delay 5, participation/entry capacity 1%, and realized cap enabled.

Fixed-session frequencies are anchored at the first executable session in the sample (session index 1, then every N sessions). Calendar week/month/quarter/bimonth schedules keep the existing US3.5 definitions. The output remains an expanded-universe Sharadar-based replay; the historical listing/category/ticker PIT completeness gate is separate and remains open.